In [2]:
# импортируем библиотеки, которые пригодятся для задачи
import torch

import torch.nn as nn

import re

import random

from datasets import load_dataset

from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import BertTokenizerFast

from tqdm import tqdm

from sklearn.model_selection import train_test_split


# Фиксируем seed для воспроизводимости
random.seed(42)
torch.manual_seed(42)

# функция для "чистки" текстов
def clean_string(text):
    # приведение к нижнему регистру
    text = text.lower()
    # удаление всего, кроме латинских букв, цифр и пробелов
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # удаление дублирующихся пробелов, удаление пробелов по краям
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# загружаем датасет WikiText-2

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")


# длины последовательностей в датасете
# seq_len = 7 => 3 токена до <MASK> + токен <MASK> + 3 токена после
seq_len = 7


# удаляем слишком короткие тексты
texts = [line for line in dataset["text"] if len(line.split()) >= seq_len]


# "чистим" тексты
cleaned_texts = list(map(clean_string, texts))


# для упрощения используем только max_texts_count текстов
max_texts_count = 7000


# разбиение на тренировочную и валидационную выборки
val_size = 0.05

train_texts, val_texts = train_test_split(cleaned_texts[:max_texts_count], test_size=val_size, random_state=42)
print(f"Train texts: {len(train_texts)}, Val texts: {len(val_texts)}")


# класс датасета
class MaskedBertDataset(Dataset):
    def __init__(self, texts, tokenizer, seq_len=7):
        self.samples = []
        for line in texts:
            token_ids = tokenizer.encode(line, add_special_tokens=False, max_length=512, truncation=True)
            if len(token_ids) < seq_len:
                continue
            for i in range(1, len(token_ids) - 1):
                context = token_ids[max(0, i - seq_len//2): i] + [tokenizer.mask_token_id] + token_ids[i+1: i+1+seq_len//2]
                if len(context) < seq_len:
                    continue
                target = token_ids[i]
                self.samples.append((context, target))
           
    def __len__(self):
        return len(self.samples)


    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x), torch.tensor(y)


# Загружаем BERT токенизатор
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")


# тренировочный и валидационный датасеты
train_dataset = MaskedBertDataset(train_texts, tokenizer, seq_len=seq_len)
val_dataset = MaskedBertDataset(val_texts, tokenizer, seq_len=seq_len)


# даталоадеры
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

Train texts: 6650, Val texts: 350


In [3]:

import torch.nn as nn


class BiRNNClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, rnn_type="GRU", combine="concat"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.combine = combine


        rnn_cls = {"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[rnn_type]
        self.rnn = rnn_cls(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)


        out_dim = hidden_dim * 2 if combine == "concat" else hidden_dim
        self.fc = nn.Linear(out_dim, vocab_size)


    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        center = x.size(1) // 2
        hidden_forward = out[:, center, :out.size(2)//2]
        hidden_backward = out[:, center, out.size(2)//2:]
        hidden_agg = hidden_forward + hidden_backward if self.combine == "sum" else torch.cat([hidden_forward, hidden_backward], dim=1)
        linear_out = self.fc(hidden_agg)
        return linear_out
    


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


vocab_size = tokenizer.vocab_size  
hidden_dim = 128

rnn_types = ["RNN", "GRU", "LSTM"]
combine_methods = ["sum", "concat"]


# Сравнение
print(f"{'RNN Type':<8} | {'Combine':<6} | {'Params':>10}")
print("-" * 35)
for rnn_type in rnn_types:
    for combine in combine_methods:
        model = BiRNNClassifier(vocab_size, hidden_dim, rnn_type, combine)
        param_count = count_parameters(model)
        print(f"{rnn_type:<8} | {combine:<6} | {param_count:>10,}")

RNN Type | Combine |     Params
-----------------------------------
RNN      | sum    |  7,910,202
RNN      | concat | 11,817,018
GRU      | sum    |  8,042,298
GRU      | concat | 11,949,114
LSTM     | sum    |  8,108,346
LSTM     | concat | 12,015,162


In [4]:

model = BiRNNClassifier(vocab_size, rnn_type="LSTM", combine="concat")
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    sum_loss = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_output = model(x_batch)
            loss = criterion(x_output, y_batch)
            preds = torch.argmax(x_output, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            sum_loss += loss.item()
    return sum_loss / len(loader), correct / total


# Основной цикл обучения
n_epochs = 3

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.
    for x_batch, y_batch in tqdm(train_loader):
        # Здесь была удалена лишняя строка присваивания
        optimizer.zero_grad()
        loss = criterion(model(x_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss, val_acc = evaluate(model, val_loader)
    
    # Дописал вывод acc, так как строка была обрезана
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f} | Val Acc: {val_acc:.3f}")

100%|██████████| 9258/9258 [14:08<00:00, 10.91it/s]


Epoch 1 | Train Loss: 6.276 | Val Loss: 5.709 | Val Acc: 0.219


 37%|███▋      | 3421/9258 [05:10<08:49, 11.02it/s]


KeyboardInterrupt: 

In [5]:

model.eval()
bad_cases, good_cases = [], []
with torch.no_grad():
    for x_batch, y_batch in val_loader:
        x_batch, y_batch = x_batch, y_batch
        logits = model(x_batch)
        preds = torch.argmax(logits, dim=1)
        for i in range(len(y_batch)):
            input_tokens = tokenizer.convert_ids_to_tokens(x_batch[i].tolist())
            true_tok = tokenizer.convert_ids_to_tokens([y_batch[i].item()])[0]
            pred_tok = tokenizer.convert_ids_to_tokens([preds[i].item()])[0]
            
            if preds[i] != y_batch[i]:
                bad_cases.append((input_tokens, true_tok, pred_tok))
            else:
                good_cases.append((input_tokens, true_tok, pred_tok))


random.seed(42)
bad_cases_sampled = random.sample(bad_cases, 5)
good_cases_sampled = random.sample(good_cases, 5)


print("\nSome incorrect predictions:")
for context, true_tok, pred_tok in bad_cases_sampled:
    print(f"Input: {' '.join(context)} | True: {true_tok} | Predicted: {pred_tok}")


print("\nSome correct predictions:")
for context, true_tok, pred_tok in good_cases_sampled:
    if true_tok == pred_tok:
        print(f"Input: {' '.join(context)} | True: {true_tok} | Predicted: {pred_tok}")


Some incorrect predictions:
Input: in a bel [MASK] holiday advertisement and | True: ##k | Predicted: in
Input: gambia s first [MASK] was against morocco | True: match | Predicted: album
Input: frame houses four [MASK] houses two brick | True: log | Predicted: or
Input: handled 250 tons [MASK] relief supplies by | True: of | Predicted: and
Input: 86 ##th minute [MASK] ##iser for york | True: equal | Predicted: and

Some correct predictions:
Input: the chelsea hierarchy [MASK] ##ton turned down | True: law | Predicted: law
Input: point system to [MASK] player if any | True: the | Predicted: the
Input: shortly transferred to [MASK] ministry of defense | True: the | Predicted: the
Input: the geological survey [MASK] canada gs ##c | True: of | Predicted: of
Input: people from keeping [MASK] burden ##some jewish | True: the | Predicted: the
